# Bounding Box Analysis

Resolution clusters, bbox geometry, Faster R-CNN resize impact, and anchor design.

In [ ]:
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from scipy.cluster.vq import kmeans2

PROJECT_ROOT = Path.cwd().parent

train = pd.read_csv(PROJECT_ROOT / "data/processed/train_annotations.csv")
test = pd.read_csv(PROJECT_ROOT / "data/processed/test_annotations.csv")

CLASSES = sorted(train["category_name"].dropna().unique())
CMAP = plt.colormaps["tab20"].resampled(len(CLASSES))
CLASS_COLOURS = {name: CMAP(i) for i, name in enumerate(CLASSES)}

train_imgs = train.groupby("image_id")[["width", "height", "file_name"]].first().reset_index()
test_imgs = test.groupby("image_id")[["width", "height"]].first().reset_index()

ann = train.dropna(subset=["annotation_id"]).copy()
ann["rel_w"] = ann["bbox_w"] / ann["width"]
ann["rel_h"] = ann["bbox_h"] / ann["height"]
ann["rel_area"] = (ann["bbox_w"] * ann["bbox_h"]) / (ann["width"] * ann["height"])
ann["aspect_ratio"] = ann["bbox_w"] / ann["bbox_h"]

print(f"{len(ann)} annotations, {ann['image_id'].nunique()} images, {len(CLASSES)} classes")

## Resolution Clusters

Proxy for device identity — distinct resolutions likely correspond to different microscopes.

In [ ]:
train_res = train_imgs.groupby(["width", "height"]).size().reset_index(name="count")
test_res = test_imgs.groupby(["width", "height"]).size().reset_index(name="count")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, res_df) in zip(axes, [("Train", train_res), ("Test", test_res)]):
    ax.scatter(res_df["width"], res_df["height"], s=res_df["count"] * 0.5, alpha=0.7)
    for _, row in res_df.iterrows():
        ax.annotate(f"{row['count']}", (row["width"], row["height"]),
                    textcoords="offset points", xytext=(8, 4), fontsize=8)
    ax.set_xlabel("Width (px)")
    ax.set_ylabel("Height (px)")
    ax.set_title(f"{name} — {len(res_df)} unique resolutions")
plt.tight_layout()
plt.show()

train_only = set(map(tuple, train_res[["width", "height"]].values)) - set(map(tuple, test_res[["width", "height"]].values))
if train_only:
    print(f"Resolutions in train but NOT test: {train_only}")

## Aspect Ratio

Directly informs anchor ratio selection.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(ann["aspect_ratio"], bins=50, edgecolor="black", alpha=0.7)
axes[0].axvline(x=1.0, color="red", linestyle="--", label="square")
axes[0].set_xlabel("Aspect ratio (w/h)")
axes[0].set_title("Bbox aspect ratio")
axes[0].legend()

data = [ann.loc[ann["category_name"] == c, "aspect_ratio"].values for c in CLASSES]
bp = axes[1].boxplot(data, vert=False, tick_labels=CLASSES, patch_artist=True)
for patch, cls in zip(bp["boxes"], CLASSES):
    patch.set_facecolor(CLASS_COLOURS[cls])
axes[1].axvline(x=1.0, color="red", linestyle="--")
axes[1].set_xlabel("Aspect ratio (w/h)")
axes[1].set_title("Aspect ratio by class")

plt.tight_layout()
plt.show()

print(f"Median: {ann['aspect_ratio'].median():.2f}, "
      f"range: {ann['aspect_ratio'].min():.2f}–{ann['aspect_ratio'].max():.2f}")

## Intra-class Variation

How much bbox size and shape vary within each class — higher variation means fixed anchors fit worse.

In [ ]:
variation = ann.groupby("category_name").agg(
    rel_area_std=("rel_area", "std"),
    aspect_ratio_std=("aspect_ratio", "std"),
    rel_w_std=("rel_w", "std"),
).reindex(CLASSES)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colours = [CLASS_COLOURS[c] for c in CLASSES]

for ax, (col, label) in zip(axes, [
    ("rel_area_std", "Std of area (%)"),
    ("aspect_ratio_std", "Std of aspect ratio"),
    ("rel_w_std", "Std of width (%)"),
]):
    vals = variation[col].values * (100 if "area" in col or "w" in col else 1)
    ax.barh(CLASSES, vals, color=colours)
    ax.set_xlabel(label)

plt.suptitle("Intra-class variation (higher = harder to detect with fixed anchors)")
plt.tight_layout()
plt.show()

## Bbox Position

Where in the frame do eggs appear? Informs whether centre-biased augmentation is appropriate.

In [ ]:
ann["centre_x"] = (ann["bbox_x"] + ann["bbox_w"] / 2) / ann["width"]
ann["centre_y"] = (ann["bbox_y"] + ann["bbox_h"] / 2) / ann["height"]

fig, ax = plt.subplots(figsize=(7, 7))
ax.hist2d(ann["centre_x"], ann["centre_y"], bins=40, cmap="hot")
ax.set_xlabel("Normalised x")
ax.set_ylabel("Normalised y")
ax.set_title("Bbox centre positions")
ax.set_aspect("equal")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Edge Annotations

Bboxes near image borders — informs crop/pad augmentation strategy.

In [ ]:
EDGE_MARGIN = 5

edge = (
    (ann["bbox_x"] < EDGE_MARGIN) |
    (ann["bbox_y"] < EDGE_MARGIN) |
    (ann["bbox_x"] + ann["bbox_w"] > ann["width"] - EDGE_MARGIN) |
    (ann["bbox_y"] + ann["bbox_h"] > ann["height"] - EDGE_MARGIN)
)

n_edge = edge.sum()
print(f"Bboxes within {EDGE_MARGIN}px of border: {n_edge} / {len(ann)} ({n_edge / len(ann) * 100:.1f}%)")

if n_edge > 0:
    edge_by_class = ann.loc[edge, "category_name"].value_counts().reindex(CLASSES, fill_value=0)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(CLASSES, edge_by_class.values, color=[CLASS_COLOURS[c] for c in CLASSES])
    ax.set_xlabel("Edge-touching annotations")
    ax.set_title(f"Edge annotations by class ({n_edge} total)")
    for i, v in enumerate(edge_by_class.values):
        if v > 0:
            ax.text(v + 0.2, i, str(v), va="center", fontsize=9)
    plt.tight_layout()
    plt.show()

## Faster R-CNN Resize Analysis

Post-resize bbox widths at candidate input resolutions.
Scale = min(target_short / min(W, H), max_long / max(W, H)).
Red lines mark danger thresholds at 10, 15, and 20 pixels.

In [ ]:
RESIZE_CONFIGS = [(800, 1333), (1024, 1600), (1333, 2000)]

for short, max_long in RESIZE_CONFIGS:
    scale = np.minimum(
        short / np.minimum(ann["width"], ann["height"]),
        max_long / np.maximum(ann["width"], ann["height"]),
    )
    ann[f"post_w_{short}"] = ann["bbox_w"] * scale

# Per-class histograms
x_max = max(ann[f"post_w_{s}"].max() for s, _ in RESIZE_CONFIGS)
fig, axes = plt.subplots(len(CLASSES), len(RESIZE_CONFIGS),
                         figsize=(4 * len(RESIZE_CONFIGS), 1.8 * len(CLASSES)))

for row, cls in enumerate(CLASSES):
    cls_data = ann[ann["category_name"] == cls]
    for ci, (short, max_long) in enumerate(RESIZE_CONFIGS):
        ax = axes[row, ci]
        ax.hist(cls_data[f"post_w_{short}"], bins=30, color=CLASS_COLOURS[cls],
                edgecolor="black", alpha=0.7)
        for thresh, ls in [(10, ":"), (15, "--"), (20, "-")]:
            ax.axvline(x=thresh, color="red", linestyle=ls, alpha=0.6)
        ax.set_xlim(0, x_max)
        if row == 0:
            ax.set_title(f"{short}/{max_long}")
        if ci == 0:
            ax.set_ylabel(cls, fontsize=8, rotation=0, labelpad=80, ha="right")
        if row < len(CLASSES) - 1:
            ax.set_xticklabels([])

axes[-1, 1].set_xlabel("Post-resize bbox width (px)")
plt.suptitle("Post-resize bbox width by class (red: 10, 15, 20px)", y=1.01)
plt.tight_layout()
plt.show()

# Summary table
rows = []
for short, max_long in RESIZE_CONFIGS:
    for cls in CLASSES:
        vals = ann.loc[ann["category_name"] == cls, f"post_w_{short}"]
        rows.append({
            "input": f"{short}/{max_long}", "class": cls,
            "min": vals.min(), "p5": vals.quantile(0.05),
            "median": vals.median(),
            "%<20px": (vals < 20).mean() * 100,
            "%<15px": (vals < 15).mean() * 100,
        })
summary = pd.DataFrame(rows)
problem = summary[(summary["%<20px"] > 0) | (summary["%<15px"] > 0)]
if len(problem) > 0:
    print("Classes with bboxes below thresholds:")
    print(problem.to_string(index=False, float_format="%.1f"))
else:
    print("No bboxes below 20px at any candidate resolution.")

# Per resolution cluster
ann["res_cluster"] = ann["width"].astype(str) + "x" + ann["height"].astype(str)
cluster_rows = []
for short, max_long in RESIZE_CONFIGS:
    col_name = f"post_w_{short}"
    for cluster, group in ann.groupby("res_cluster"):
        pct_20 = (group[col_name] < 20).mean() * 100
        pct_15 = (group[col_name] < 15).mean() * 100
        if pct_20 > 0 or pct_15 > 0:
            cluster_rows.append({
                "input": f"{short}/{max_long}", "cluster": cluster,
                "n": len(group), "%<20px": pct_20, "%<15px": pct_15,
            })
if cluster_rows:
    print()
    print("Resolution clusters with small post-resize bboxes:")
    print(pd.DataFrame(cluster_rows).to_string(index=False, float_format="%.1f"))
else:
    print()
    print("No resolution clusters produce dangerously small bboxes.")

## Anchor Box Clustering

K-means on relative bbox dimensions to find natural anchor shapes.

In [ ]:
wh = np.column_stack([ann["rel_w"].values, ann["rel_h"].values]).astype(np.float32)

results = {}
for k in range(2, 7):
    centroids, labels = kmeans2(wh, k, minit="++", iter=20)
    assigned = centroids[labels]
    inter_w = np.minimum(wh[:, 0], assigned[:, 0])
    inter_h = np.minimum(wh[:, 1], assigned[:, 1])
    inter = inter_w * inter_h
    union = wh[:, 0] * wh[:, 1] + assigned[:, 0] * assigned[:, 1] - inter
    results[k] = {"centroids": centroids, "mean_iou": (inter / union).mean()}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ks = sorted(results)
axes[0].plot(ks, [results[k]["mean_iou"] for k in ks], "o-")
axes[0].set_xlabel("k (number of anchors)")
axes[0].set_ylabel("Mean IoU with nearest anchor")
axes[0].set_title("Anchor clustering — elbow plot")

best_k = 5
centroids = results[best_k]["centroids"]
axes[1].scatter(wh[:, 0], wh[:, 1], alpha=0.1, s=5, c="grey")
axes[1].scatter(centroids[:, 0], centroids[:, 1], c="red", s=200, marker="X",
                edgecolors="black", zorder=5)
for cx, cy in centroids:
    axes[1].add_patch(patches.Rectangle(
        (cx - cx / 2, cy - cy / 2), cx, cy,
        linewidth=2, edgecolor="red", facecolor="none", linestyle="--",
    ))
axes[1].set_xlabel("Relative width")
axes[1].set_ylabel("Relative height")
axes[1].set_title(f"k={best_k} anchor centroids")

plt.tight_layout()
plt.show()

print(f"Anchor centroids (k={best_k}, relative w x h):")
for i, (w, h) in enumerate(sorted(centroids, key=lambda x: x[0] * x[1])):
    print(f"  {i + 1}: {w:.3f} x {h:.3f}  (aspect ratio {w / h:.2f})")
print(f"Mean IoU: {results[best_k]['mean_iou']:.3f}")

## Outlier Inspection

The 1546x505 image — only one at that resolution.

In [ ]:
outlier = train[(train["width"] == 1546) & (train["height"] == 505)]
if len(outlier) > 0:
    r = outlier.iloc[0]
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.imshow(Image.open(PROJECT_ROOT / r["file_path"]))
    for _, row in outlier.iterrows():
        if pd.notna(row.get("category_name")):
            c = CLASS_COLOURS[row["category_name"]]
            ax.add_patch(patches.Rectangle(
                (row["bbox_x"], row["bbox_y"]), row["bbox_w"], row["bbox_h"],
                linewidth=2, edgecolor=c, facecolor="none",
            ))
    ax.set_title(f"{r['file_name']} — {r['width']}x{r['height']} — {r['category_name']}")
    ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Outlier image not found.")